# Mercari Price Regression — runnable notebook

This notebook is a cell-by-cell version of the provided scripts:

- `dataset.py` (dataset + text building)
- `model.py` (transformer + regression head)
- `evaluate.py` (RMSE/MAE and prediction helpers)
- `train.py` (training loop and config)

It’s designed so you can run sequentially, tweak config, and re-train/evaluate/predict.


## 0) Imports + basic setup


In [ ]:
# 1) Imports + basic setup
import os

# Must be set BEFORE CUDA initializes — required for deterministic matmul
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import json
import math
import random
from dataclasses import dataclass, asdict
from typing import Dict, Optional, Any

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

torch: 2.10.0+cu128
cuda available: True
device: NVIDIA A100-SXM4-80GB


## 1) Reproducibility helpers + device


In [ ]:
# 2) Reproducibility helpers + device
def set_seed(seed: int = 42):
    """Set random seeds and enable deterministic mode across all backends.

    Covers Python, NumPy, and PyTorch (CPU + CUDA). Also disables cuDNN
    benchmarking and enables deterministic algorithms to maximise
    run-to-run reproducibility.

    Args:
        seed: The seed value to use across all backends.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    torch.use_deterministic_algorithms(True, warn_only=True)


set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

## 3) Dataset utilities (from `dataset.py`)


In [ ]:
# 3) Dataset

def build_text(desc) -> str:
    """Description-only input. Empty / NaN descriptions become empty strings."""
    if pd.isna(desc):
        return ""
    return str(desc).strip()


class MercariTextDataset(Dataset):
    """
    Description-only dataset for Mercari price regression.

    Reads item_description and produces pre-tokenized tensors + log1p(price)
    labels. The variant (Original / P1 / P2) is determined by which CSV is
    loaded upstream — the column name is always 'item_description'.

    Pre-tokenization happens once in __init__ so __getitem__ is a pure lookup.
    """
    TEXT_COL = "item_description"
    PRICE_COL = "price"

    def __init__(
        self,
        df: pd.DataFrame,
        tokenizer,
        max_length: int = 256,
        has_labels: bool = True,
    ):
        if self.TEXT_COL not in df.columns:
            raise ValueError(
                f"'{self.TEXT_COL}' not in df columns. Available: {list(df.columns)}"
            )

        self.df = df.reset_index(drop=True)
        self.max_length = max_length
        self.has_labels = has_labels

        # Build texts once
        texts = [build_text(d) for d in self.df[self.TEXT_COL]]

        # Tokenize once with fixed padding so DataLoader can stack tensors
        enc = tokenizer(
            texts,
            truncation=True,
            max_length=max_length,
            padding="max_length",
            return_attention_mask=True,
        )
        self.input_ids = enc["input_ids"]
        self.attention_mask = enc["attention_mask"]

        if has_labels:
            if self.PRICE_COL not in self.df.columns:
                raise ValueError(f"has_labels=True but '{self.PRICE_COL}' not in df")
            prices = self.df[self.PRICE_COL].astype(float).values
            if (prices < 0).any():
                raise ValueError("Negative prices found — log1p will produce NaN")
            # Train on log1p(price) so MSE loss equals MSLE — the squared term
            # inside our primary metric RMSLE.
            self.labels = np.log1p(prices).astype(np.float32)
        else:
            self.labels = None

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        item = {
            "input_ids": torch.tensor(self.input_ids[idx], dtype=torch.long),
            "attention_mask": torch.tensor(self.attention_mask[idx], dtype=torch.long),
        }
        if self.has_labels:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float32)
        return item

## 4) Model (from `model.py`)


In [ ]:
from dataclasses import dataclass

@dataclass
class ModelOutput:
    preds: torch.Tensor
    loss: Optional[torch.Tensor]


class PriceRegressor(nn.Module):
    """
    Fine-tunes transformer + regression head.
    Input: tokenized text (name + description)
    Output: log1p(price)
    """
    def __init__(self, encoder_name: str, dropout: float = 0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(encoder_name)
        h = self.encoder.config.hidden_size
        # NOTE: encoder is intentionally NOT frozen — this is the fine-tuned variant.

        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(h, 1),
        )
        self.loss_fn = nn.MSELoss()

    def forward(self, input_ids, attention_mask, labels=None) -> ModelOutput:
        """Run a forward pass through the frozen encoder and linear head.

        Extracts the [CLS] embedding without gradient tracking, then
        passes it through the trainable dropout + linear head to produce
        a scalar log-price prediction per sample.

        Args:
            input_ids: Tokenised input tensor of shape (batch, seq_len).
            attention_mask: Attention mask tensor of shape (batch, seq_len).

        Returns:
            Tensor of shape (batch,) with predicted log-prices.
        """
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]          # [B, H]
        pred = self.head(cls).squeeze(-1)             # [B]

        loss = None
        if labels is not None:
            loss = self.loss_fn(pred, labels)

        return ModelOutput(preds=pred, loss=loss)


## 5) Evaluation helpers (from `evaluate.py`)


In [ ]:
@torch.no_grad()
def evaluate_loader(model, loader, device: torch.device) -> Dict[str, float]:
    """
    Requires labels to exist in loader batches.
    Computes RMSE/MAE on original price scale by expm1.
    """
    model.eval()

    preds_log, labels_log = [], []

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        out = model(input_ids=input_ids, attention_mask=attention_mask, labels=None)

        preds_log.append(out.preds.detach().cpu().numpy())
        labels_log.append(labels.detach().cpu().numpy())

    preds_log = np.concatenate(preds_log)
    labels_log = np.concatenate(labels_log)

    mse_log = float(np.mean((preds_log - labels_log) ** 2))

    rmsle = float(np.sqrt(mse_log))

    preds = np.expm1(preds_log)
    labels = np.expm1(labels_log)

    rmse = float(np.sqrt(np.mean((preds - labels) ** 2)))
    mae = float(np.mean(np.abs(preds - labels)))

    return {"mse_log": mse_log,
            "rmse": rmse,
            "mae": mae,
            "rmsle": rmsle}


@torch.no_grad()
def predict_loader(model, loader, device: torch.device) -> np.ndarray:
    """
    Works even if loader has no labels (e.g., your test file).
    Returns predictions on original price scale.
    """
    model.eval()
    preds_log = []

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        out = model(input_ids=input_ids, attention_mask=attention_mask, labels=None)
        preds_log.append(out.preds.detach().cpu().numpy())

    preds_log = np.concatenate(preds_log)
    return np.expm1(preds_log)


## 6) Config (from `train.py`)


In [ ]:
@dataclass
class TrainConfig:
    encoder_name: str = "distilbert-base-uncased"
    max_length: int = 256

    lr: float = 5e-5
    weight_decay: float = 0.01
    batch_size: int = 32
    num_epochs: int = 3
    warmup_ratio: float = 0.06
    grad_clip: float = 1.0

    num_workers: int = 2
    seed: int = 42
    out_dir: str = "checkpoints/price_model"

cfg = TrainConfig()
cfg


TrainConfig(encoder_name='distilbert-base-uncased', max_length=256, lr=5e-05, weight_decay=0.01, batch_size=32, num_epochs=3, warmup_ratio=0.06, grad_clip=1.0, num_workers=2, seed=42, out_dir='checkpoints/price_model')

## Load Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:

os.listdir("/content/drive/MyDrive/Lokaverkefni - Gögn")

['train.csv',
 'validation_sample.csv',
 'train_category_distribution.csv',
 'validation_category_distribution.csv',
 'test_category_distribution.csv',
 'train_sample.csv',
 'test_sample.csv',
 'Normalized Prompt 2 Data',
 'Normalized Prompt 1 Data',
 'Normalized Prompt 3 Data',
 'Test Predictions',
 'Predictions',
 'Normalized P1 Description Only']

## 7) Load CSVs + minimal cleaning (matches `train.py`)


In [ ]:
import shutil

# filenames and path modified to what data is being trained and tested
BASE_PATH = "/content/drive/MyDrive/Lokaverkefni - Gögn/Normalized P1 Description Only"

for fname in ["train_p1_desc_only.csv", "validation_p1_desc_only.csv", "test_p1_desc_only.csv"]:
    src = os.path.join(BASE_PATH, fname)

train_df = pd.read_csv(os.path.join(BASE_PATH, "train_p1_desc_only.csv"))
val_df   = pd.read_csv(os.path.join(BASE_PATH, "validation_p1_desc_only.csv"))
test_df  = pd.read_csv(os.path.join(BASE_PATH, "test_p1_desc_only.csv"))

print("raw shapes:", train_df.shape, val_df.shape, test_df.shape)
train_df.head()

raw shapes: (85675, 5) (18359, 5) (18360, 5)


,train_id,name,category_name,price,item_description
0,1288696,Rae Dunn Mixing Bowls Reserved K. Cannon,Home/Kitchen & Dining/Dining & Entertaining,100.0,Reserved for Kim Cannon.
1,1016811,2 4x4 monogrammed decal/sticker,Handmade/Paper Goods/Sticker,8.0,Monogram decal available in any size. Please c...
2,5636,CINDY CRAWFORD MEANINGFUL BEAUTY,Beauty/Skin Care/Face,30.0,Cindy Crawford Meaningful Beauty five-piece to...
3,1469134,Simply Southern,Women/Tops & Blouses/T-Shirts,18.0,"T-shirts, size small. Barely worn. Card holder..."
4,455146,⚡️NWT Just do it Leggings Size Medium,"Women/Athletic Apparel/Pants, Tights, Leggings",34.0,New with tag Nike Camo Leg-A-See. The original...


In [ ]:
cols = ["item_description", "price", "train_id"]
train_df = train_df[cols]
val_df   = val_df[cols]
test_df  = test_df[cols]

train_df = train_df[train_df["price"] > 0].reset_index(drop=True)
val_df   = val_df[val_df["price"] > 0].reset_index(drop=True)
test_df  = test_df[test_df["price"] > 0].reset_index(drop=True)


## 8) Tokenizer + Datasets + DataLoaders


In [ ]:
set_seed(cfg.seed)

tokenizer = AutoTokenizer.from_pretrained(cfg.encoder_name)

train_ds = MercariTextDataset(train_df, tokenizer, max_length=cfg.max_length, has_labels=True)
val_ds   = MercariTextDataset(val_df, tokenizer, max_length=cfg.max_length, has_labels=True)
test_ds = MercariTextDataset(test_df, tokenizer, max_length=cfg.max_length, has_labels=False)

train_loader = DataLoader(
    train_ds, batch_size=cfg.batch_size, shuffle=True,
    num_workers=cfg.num_workers, pin_memory=(device.type == "cuda")
)
val_loader = DataLoader(
    val_ds, batch_size=cfg.batch_size, shuffle=False,
    num_workers=cfg.num_workers, pin_memory=(device.type == "cuda")
)
test_loader = DataLoader(
    test_ds, batch_size=cfg.batch_size, shuffle=False,
    num_workers=cfg.num_workers, pin_memory=(device.type == "cuda")
)

len(train_loader), len(val_loader), len(test_loader)


(2676, 574, 574)

## 9) Model + optimizer + scheduler


In [ ]:
model = PriceRegressor(cfg.encoder_name).to(device).float()

trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=cfg.lr, weight_decay=cfg.weight_decay)

total_steps = cfg.num_epochs * len(train_loader)
warmup_steps = int(cfg.warmup_ratio * total_steps)
scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
total_steps, warmup_steps


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_827/2674537954.py:10: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))


(8028, 481)

In [ ]:
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)

print("Trainable params:", trainable)
print("Frozen params:", frozen)

Trainable params: 66363649
Frozen params: 0


## 10) Training loop (matches `train.py`) — saves best checkpoint


In [ ]:
os.makedirs(cfg.out_dir, exist_ok=True)

with open(os.path.join(cfg.out_dir, "train_config.json"), "w") as f:
    json.dump(asdict(cfg), f, indent=2)

best_val_rmsle = float("inf")
history = []
history_path = os.path.join(cfg.out_dir, "history.csv")

for epoch in range(1, cfg.num_epochs + 1):
    model.train()
    running = 0.0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{cfg.num_epochs}", leave=False)
    for batch_idx, batch in enumerate(pbar, start=1):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
             out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
             loss = out.loss

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        running += float(loss.item())
        avg_so_far = running / batch_idx
        pbar.set_postfix(train_mse=f"{avg_so_far:.4f}", lr=f"{optimizer.param_groups[0]['lr']:.2e}")

    avg_loss = running / max(1, len(train_loader))
    val_metrics = evaluate_loader(model, val_loader, device=device)
    val_rmsle = val_metrics["rmsle"]

    current_lr = optimizer.param_groups[0]["lr"]
    history.append({
        "epoch": epoch,
        "train_mse": avg_loss,
        "val_rmse": val_metrics["rmse"],
        "val_mae": val_metrics["mae"],
        "val_mse_log": val_metrics["mse_log"],
        "val_rmsle": val_metrics["rmsle"],
        "lr": current_lr,
    })
    pd.DataFrame(history).to_csv(history_path, index=False)

    print(
        f"Epoch {epoch}/{cfg.num_epochs} | "
        f"train_mse={avg_loss:.4f} | val_rmsle={val_rmsle:.4f} | val_mae={val_metrics['mae']:.4f}"
    )

    if val_rmsle < best_val_rmsle:
        best_val_rmsle = val_rmsle
        ckpt_path = os.path.join(cfg.out_dir, "best.pt")
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "encoder_name": cfg.encoder_name,
                "best_val_rmsle": best_val_rmsle,
                "epoch": epoch,
            },
            ckpt_path,
        )
        print(f"  ✅ saved best -> {ckpt_path}")

print(f"Done. Best val RMSLE: {best_val_rmsle:.4f}")


Epoch 1/3:   0%|          | 0/2676 [00:00<?, ?it/s]

/tmp/ipykernel_827/3138709174.py:22: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
/tmp/ipykernel_827/3138709174.py:31: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


Epoch 1/3 | train_mse=0.5468 | val_rmsle=0.5754 | val_mae=13.0017
  ✅ saved best -> checkpoints/price_model/best.pt


Epoch 2/3:   0%|          | 0/2676 [00:00<?, ?it/s]

Epoch 2/3 | train_mse=0.2774 | val_rmsle=0.5557 | val_mae=12.2876
  ✅ saved best -> checkpoints/price_model/best.pt


Epoch 3/3:   0%|          | 0/2676 [00:00<?, ?it/s]

Epoch 3/3 | train_mse=0.2026 | val_rmsle=0.5593 | val_mae=12.5641
Done. Best val RMSLE: 0.5557


\## 11) Load best checkpoint (optional) + re-evaluate


In [ ]:
ckpt_path = os.path.join(cfg.out_dir, "best.pt")
if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    print("Loaded:", ckpt_path, "| best_val_rmsle:", ckpt.get("best_val_rmsle"), "| epoch:", ckpt.get("epoch"))
else:
    print("No checkpoint found at", ckpt_path)

evaluate_loader(model, val_loader, device=device)


Loaded: checkpoints/price_model/best.pt | best_val_rmsle: 0.5557396540654759 | epoch: 2


{'mse_log': 0.3088465631008148,
 'rmse': 32.893245697021484,
 'mae': 12.287631034851074,
 'rmsle': 0.5557396540654759}

## 12) Predict on test set + save submission-style CSV


In [ ]:
pred_prices = predict_loader(model, test_loader, device=device)

out_pred_path = os.path.join(cfg.out_dir, "test_predictions.csv")

out_df = pd.DataFrame({
    "train_id":   test_df["train_id"].values,
    "price_pred": pred_prices,
})

out_df.to_csv(out_pred_path, index=False)
out_df.head(), out_pred_path

(   train_id  price_pred
 0    450414   13.713344
 1    406368   21.174320
 2    136022    7.570452
 3    596088   25.242237
 4   1161685    5.087439,
 'checkpoints/price_model/test_predictions.csv')

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

actual = test_df["price"].values
# pred_prices = predict_loader(model, test_loader, device=device)

rmse = np.sqrt(mean_squared_error(actual, pred_prices))
mae = mean_absolute_error(actual, pred_prices)
median_ae = np.median(np.abs(actual - pred_prices))

#RMSLE — requires non-negative values
rmsle = np.sqrt(np.mean((np.log1p(np.clip(pred_prices, 0, None)) - np.log1p(actual)) ** 2))

print(f"RMSE:      {rmse:.4f}")
print(f"MAE:       {mae:.4f}")
print(f"Median AE: {median_ae:.4f}")
print(f"RMSLE:     {rmsle:.4f}")

RMSE:      31.3397
MAE:       12.3573
Median AE: 5.4863
RMSLE:     0.5546
